In [0]:
%run ./01_extracao_produtos_categorias

In [0]:
from pyspark.sql import functions as F

def analisar_qualidade_dataframe(df, nome):
    if df is None:
        print(f"⚠️ {nome}: dados ainda não carregados.")
        return

    print(f"========================================")
    print(f"  Qualidade dos Dados: {nome}")
    print(f"========================================")

    # 1. Contagem de Nulos por Coluna
    print("\n🔍 Valores nulos por coluna:")
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show(truncate=False)

    # 2. Verificação de Registros Totais e Duplicados
    total_registros = df.count()
    registros_unicos = df.distinct().count()
    duplicados = total_registros - registros_unicos

    print("\n📊 Análise de Duplicidades:")
    print(f"  • Total de registros:  {total_registros:,}")
    print(f"  • Registros únicos:    {registros_unicos:,}")
    print(f"  • Linhas duplicadas:   {duplicados:,}")

    # 3. Exibir Schema para validação de tipos
    print("\n📋 Tipos de Dados (Schema):")
    df.printSchema()

# Executando a auditoria para validação
analisar_qualidade_dataframe(df_produtos, "Produtos")
analisar_qualidade_dataframe(df_categorias, "Categorias")
analisar_qualidade_dataframe(df_clientes, "Clientes")
analisar_qualidade_dataframe(df_pedidos, "Pedidos")

---
## Impacto da Ausência de Deduplicação nos KPIs de Pedidos

Quando o mesmo arquivo/lote é processado mais de uma vez **sem controle de estado**, os registros duplicados **incham os KPIs de volume** — total de pedidos, receita e ticket médio ficam artificialmente maiores. Abaixo simulamos esse cenário comparando os KPIs reais vs. reprocessados (1x, 2x e 3x).

In [0]:
# ============================================================================
# Impacto da ausência de deduplicação nos KPIs de ecommerce_pedidos
# ============================================================================
# Lê a tabela ecommerce_pedidos via Lakehouse Federation e simula o que
# aconteceria se o mesmo conjunto de dados fosse processado 1x, 2x e 3x
# SEM o controle de estado (hash) implementado no notebook 05.
# df_pedidos já está disponível via %run ./01_extracao_produtos_categorias

from pyspark.sql import functions as F

def calcular_kpis(df, rotulo):
    """Calcula KPIs de volume de pedidos para um DataFrame."""
    total = df.count()
    valor_total = df.agg(F.sum("valor_total")).collect()[0][0] or 0
    ticket_medio = df.agg(F.avg("valor_total")).collect()[0][0] or 0
    return (rotulo, total, valor_total, ticket_medio)

# KPIs reais (1 processamento)
kpi_1x = calcular_kpis(df_pedidos, "1x (correto)")

# Simulação de reprocessamento: union consigo mesmo (2x e 3x)
df_2x = df_pedidos.union(df_pedidos)
df_3x = df_pedidos.union(df_pedidos).union(df_pedidos)

kpi_2x = calcular_kpis(df_2x, "2x (sem dedup)")
kpi_3x = calcular_kpis(df_3x, "3x (sem dedup)")

# Tabela comparativa
df_comparacao = spark.createDataFrame(
    [kpi_1x, kpi_2x, kpi_3x],
    ["cenario", "total_pedidos", "valor_total", "ticket_medio"]
)

print("📊 Comparação de KPIs: correto vs. reprocessado sem deduplicação\n")
df_comparacao.show(truncate=False)

In [0]:
# ============================================================================
# Detalhamento: inflação por status de pedido
# ============================================================================
# Mostra como o reprocessamento multiplica a contagem em cada status.

print("📊 Pedidos por status — 1x (correto) vs 3x (sem dedup):\n")

# Contagem correta por status
kpi_status_1x = df_pedidos.groupBy("status_pedido") \
    .agg(F.count("*").alias("pedidos_1x")) \
    .orderBy(F.desc("pedidos_1x"))

# Contagem inflada (3x)
kpi_status_3x = df_3x.groupBy("status_pedido") \
    .agg(F.count("*").alias("pedidos_3x")) \
    .orderBy(F.desc("pedidos_3x"))

# Join para comparar lado a lado
df_status_comp = kpi_status_1x.join(kpi_status_3x, "status_pedido", "inner") \
    .withColumn("inflacao_pct", F.round((F.col("pedidos_3x") / F.col("pedidos_1x") - 1) * 100, 0)) \
    .select("status_pedido", "pedidos_1x", "pedidos_3x", "inflacao_pct")

df_status_comp.show(truncate=False)

# Resumo final
print("\n💡 Conclusão:")
print(f"   Sem deduplicação, processar 3x o mesmo arquivo multiplica os KPIs por 3:")
print(f"   • Total de pedidos: {kpi_1x[1]:,} → {kpi_3x[1]:,}  (+200%)")
print(f"   • Valor total:      R$ {kpi_1x[2]:,.2f} → R$ {kpi_3x[2]:,.2f}  (+200%)")
print(f"   • Ticket médio:     R$ {kpi_1x[3]:,.2f} → R$ {kpi_3x[3]:,.2f}  (sem alteração — média não muda)")
print(f"\n   ⚠️  O controle de estado por hash (notebook 05_filtragem_pedidos_controle)")
print(f"   evita esse problema: registros já processados são ignorados.")

In [0]:
# ============================================================================
# Configuracao JDBC — .env dinamico + funcoes de leitura segura
# ============================================================================

from dotenv import load_dotenv
from pathlib import Path
import os

def descobrir_env_via_contexto_notebook():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = ctx.notebookPath().get()
        pasta_notebook = "/".join(notebook_path.split("/")[:-1])
        return f"/Workspace{pasta_notebook}/.env"
    except Exception:
        env_found = next(Path("/Workspace").rglob(".env"), None)
        return str(env_found) if env_found else None

dotenv_path = descobrir_env_via_contexto_notebook()

if dotenv_path and os.path.exists(dotenv_path):
    load_dotenv(dotenv_path)
    print(f"✅ .env carregado de: {dotenv_path}")
else:
    print("❌ .env nao encontrado.")

sql_host = os.getenv("SQL_HOST").strip('"').strip("'")
sql_database = os.getenv("SQL_DATABASE").strip('"').strip("'")
sql_username = os.getenv("SQL_USERNAME").strip('"').strip("'")
sql_password = os.getenv("SQL_PASSWORD").strip('"').strip("'")

def ler_tabela_jdbc(schema_tabela):
    return (
        spark.read.format("sqlserver")
        .option("host", sql_host).option("port", 1433)
        .option("database", sql_database).option("dbtable", schema_tabela)
        .option("user", sql_username).option("password", sql_password)
        .option("encrypt", "true").option("trustServerCertificate", "false")
        .option("readOnly", "true").load()
    )

def bloquear_escrita_jdbc():
    raise PermissionError("🔒 ESCRITA BLOQUEADA! Use workspace.default.<tabela> (Delta) para gravar.")

print("🔒 Conexao JDBC configurada como SOMENTE LEITURA")

In [0]:
# ============================================================================
# Leitura e analise de dbo.ecommerce_clientes via JDBC
# ============================================================================

df_clientes_jdbc = ler_tabela_jdbc("dbo.ecommerce_clientes")

print(f"✅ Tabela lida via JDBC (readOnly=true): {df_clientes_jdbc.count()} linhas")
print("\n=== ESTRUTURA DOS DADOS ===")
df_clientes_jdbc.printSchema()
print("\n=== AMOSTRA DOS DADOS ===")
display(df_clientes_jdbc.limit(10))

In [0]:
# ============================================================================
# Validacao de volume e integridade da chave primaria
# ============================================================================

quantidade_linhas = df_clientes_jdbc.count()
quantidade_colunas = len(df_clientes_jdbc.columns)

print("=== VOLUME DE DADOS ===")
print(f"Linhas   : {quantidade_linhas}")
print(f"Colunas  : {quantidade_colunas}")

nulos_id_cliente = df_clientes_jdbc.filter(F.col("id_cliente").isNull()).count()

print("\n=== INTEGRIDADE DA CHAVE PRIMARIA ===")
print(f"Registros com id_cliente nulo: {nulos_id_cliente}")

if nulos_id_cliente == 0:
    print("Status: saudavel - nenhum registro com id_cliente nulo.")
else:
    print("Status: atencao - existem registros com id_cliente nulo.")

print("\n💡 Comparacao: Lakehouse Federation vs JDBC")
print(f"   Lakehouse (df_clientes): {df_clientes.count()} linhas")
print(f"   JDBC (df_clientes_jdbc): {quantidade_linhas} linhas")
if df_clientes.count() == quantidade_linhas:
    print("   ✅ Ambos retornam o mesmo numero de linhas!")